# Notebook 6 — Capital Allocation: NPV, IRR & Decisions

### 📺 Brought to you by **Christian Martinez: AI for Finance**
👉 **Subscribe here: https://www.youtube.com/@christianmartinezAIforFinance**

> *Python for CFOs — a 7-part journey from spreadsheet to strategy.*

---
**Level:** 🔴 Advanced  
The CFO's core job: decide where capital goes. Rank 8 projects by NPV and IRR, run payback, and build a clean capital-allocation recommendation table.

---

## Dataset: `capex_projects.csv` (8 investment projects, included)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

try:
    proj = pd.read_csv("capex_projects.csv")
except FileNotFoundError:
    proj=pd.DataFrame({
      "project":[f"Project_{c}" for c in "ABCDEFGH"],
      "initial_investment":[500000,750000,300000,1200000,450000,600000,900000,250000],
      "annual_cashflow":[140000,180000,95000,310000,120000,155000,210000,80000],
      "years":[5,6,5,7,5,6,6,4],
      "risk_score":[3,5,2,7,4,3,6,2]})
proj

### 1. Net Present Value (NPV)
Discount each project's future cashflows back to today. Positive NPV = creates value.

In [ ]:
DISCOUNT_RATE = 0.10  # 10% hurdle rate

def npv(rate, initial, annual_cf, years):
    cashflows = [-initial] + [annual_cf]*years
    return sum(cf/(1+rate)**t for t,cf in enumerate(cashflows))

proj["NPV"] = proj.apply(
    lambda r: npv(DISCOUNT_RATE, r.initial_investment, r.annual_cashflow, r.years), axis=1
).round(0)
proj[["project","initial_investment","NPV"]].sort_values("NPV", ascending=False)

### 2. Internal Rate of Return (IRR)
The discount rate where NPV = 0. Compare it to your hurdle rate.

In [ ]:
def irr(initial, annual_cf, years, guesses=np.linspace(0.001,1,1000)):
    cashflows=[-initial]+[annual_cf]*years
    best, best_npv = 0, 1e18
    for g in guesses:
        val=abs(sum(cf/(1+g)**t for t,cf in enumerate(cashflows)))
        if val<best_npv: best_npv, best = val, g
    return best

proj["IRR"] = proj.apply(
    lambda r: irr(r.initial_investment, r.annual_cashflow, r.years), axis=1)
proj["IRR_pct"] = (proj["IRR"]*100).round(1)
proj["beats_hurdle"] = proj["IRR"] > DISCOUNT_RATE
proj[["project","NPV","IRR_pct","beats_hurdle"]].sort_values("IRR_pct", ascending=False)

### 3. Payback period — how fast do we recover the cash?

In [ ]:
proj["payback_years"] = (proj["initial_investment"]/proj["annual_cashflow"]).round(1)
proj[["project","payback_years","years"]].sort_values("payback_years")

### 4. Profitability Index & risk-adjusted ranking

In [ ]:
proj["PI"] = ((proj["NPV"]+proj["initial_investment"])/proj["initial_investment"]).round(2)
# Risk-adjusted score: NPV per unit of risk
proj["risk_adj_npv"] = (proj["NPV"]/proj["risk_score"]).round(0)
ranking = proj.sort_values("risk_adj_npv", ascending=False)
ranking[["project","NPV","IRR_pct","PI","risk_score","risk_adj_npv"]]

### 5. The capital allocation chart
Bubble size = investment, color = risk. Top-right is the sweet spot.

In [ ]:
fig, ax = plt.subplots(figsize=(11,6))
sc=ax.scatter(proj["IRR_pct"], proj["NPV"],
   s=proj["initial_investment"]/2000, c=proj["risk_score"],
   cmap="RdYlGn_r", alpha=0.75, edgecolors="black")
for _,r in proj.iterrows():
    ax.annotate(r.project.replace("Project_",""), (r.IRR_pct, r.NPV),
                ha="center", va="center", fontweight="bold")
ax.axvline(DISCOUNT_RATE*100, color="red", linestyle="--", label="Hurdle 10%")
ax.set_xlabel("IRR (%)"); ax.set_ylabel("NPV ($)")
ax.set_title("Capital Allocation Map (bubble=investment, color=risk)")
plt.colorbar(sc, label="Risk Score"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 6. Budget-constrained selection
With a fixed budget, pick the mix that maximizes total NPV (greedy by efficiency).

In [ ]:
BUDGET = 2_000_000
candidates = proj[proj["beats_hurdle"]].sort_values("PI", ascending=False)

selected, spent = [], 0
for _,r in candidates.iterrows():
    if spent + r.initial_investment <= BUDGET:
        selected.append(r.project); spent += r.initial_investment

chosen = proj[proj.project.isin(selected)]
print(f"Budget: ${BUDGET:,.0f} | Allocated: ${spent:,.0f}")
print(f"Selected: {selected}")
print(f"Total NPV captured: ${chosen.NPV.sum():,.0f}")

### 🎯 Your turn
Change the hurdle rate to 14% — which projects drop out? Re-run the budget selection with $1.5M instead of $2M.

---
### ✅ Recap
- **NPV** measures value created • **IRR** vs hurdle is the go/no-go test • payback & PI add nuance • risk-adjust before ranking • optimize allocation under a budget
- **Next — Notebook 7:** the capstone — Monte Carlo risk + an AI-powered analyst.

📺 **Subscribe: https://www.youtube.com/@christianmartinezAIforFinance**